In [1]:
import pandas as pd
from autogluon.tabular import TabularPredictor

df = pd.read_csv('train.csv').drop(['project_id', 'hours_spent', 'hours_bucket'], axis=1)


In [2]:
from sklearn.metrics import root_mean_squared_log_error
from sklearn.model_selection import train_test_split
import copy, numpy as np
from autogluon.tabular.configs.hyperparameter_configs import get_hyperparameter_config

hp = copy.deepcopy(get_hyperparameter_config("commercial_2026_08_05"))
for model_name, configs in hp.items():
    for cfg in configs:
        cfg.pop("ag.min_rows", None)

train, hold = train_test_split(df, test_size=0.2, random_state=42)

predictor = TabularPredictor(label="log_hours", eval_metric="rmse", path="models/ag").fit(train, presets="high_v150", hyperparameters=hp, time_limit=1200)


pred = np.expm1(predictor.predict(hold.drop(columns=["log_hours"]))).clip(0.1)
print(root_mean_squared_log_error(np.expm1(hold["log_hours"]), pred))

Preset alias specified: 'high_v150' maps to 'high_quality_v150'.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.13.5
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26300
CPU Count:          16
Pytorch Version:    2.15.0+cu134
CUDA Version:       13.4
GPU Memory:         GPU 0: 12.00/12.00 GB
Total GPU Memory:   Free: 12.00 GB, Allocated: 0.00 GB, Total: 12.00 GB
GPU Count:          1
Memory Avail:       17.20 GB / 31.92 GB (53.9%)
Disk Space Avail:   6.06 GB / 222.68 GB (2.7%)
	We recommend a minimum available disk space of 10 GB, and large datasets may require more.
Presets specified: ['high_v150']
Stack configuration (auto_stack=True): num_stack_levels=0, num_bag_folds=8, num_bag_sets=1
Note: `save_bag_folds=False`! This will greatly reduce peak disk usage during fit (by ~8x), but runs the risk of an out-of-memory error during model refit if memory is small relative to

[1000]	valid_set's rmse: 0.844123
[2000]	valid_set's rmse: 0.843161
[1000]	valid_set's rmse: 0.770856
[1000]	valid_set's rmse: 0.830042
[1000]	valid_set's rmse: 0.764438
[2000]	valid_set's rmse: 0.763427
[3000]	valid_set's rmse: 0.761467
[1000]	valid_set's rmse: 0.835927
[1000]	valid_set's rmse: 0.829687
[2000]	valid_set's rmse: 0.823997
[3000]	valid_set's rmse: 0.823068
[4000]	valid_set's rmse: 0.82116
[5000]	valid_set's rmse: 0.82163


	-0.8164	 = Validation score   (-root_mean_squared_error)
	183.33s	 = Training   runtime
	2.63s	 = Validation runtime
Fitting model: LightGBM_r8_BAG_L1 ... Training model for up to 771.40s of the 771.40s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=8, gpus=0)


[1000]	valid_set's rmse: 0.845503
[1000]	valid_set's rmse: 0.850306
[1000]	valid_set's rmse: 0.77872
[2000]	valid_set's rmse: 0.776352
[1000]	valid_set's rmse: 0.842738


	-0.823	 = Validation score   (-root_mean_squared_error)
	6.87s	 = Training   runtime
	0.05s	 = Validation runtime
Fitting model: CatBoost_BAG_L1 ... Training model for up to 764.43s of the 764.43s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=8, gpus=0)


[1000]	valid_set's rmse: 0.840366


	-0.8268	 = Validation score   (-root_mean_squared_error)
	236.48s	 = Training   runtime
	0.1s	 = Validation runtime
Fitting model: RealMLP_r9_BAG_L1 ... Training model for up to 527.80s of the 527.80s of remaining time.
	Forcing `save_bag_folds=True` because child model does not support `refit_full`.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=8, gpus=1)
C:\Users\SilentSword\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
W1124 18:37:27.013000 5840 site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels
C:\Users\SilentSword\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespe

0.7898675591388439


In [3]:
predicator = TabularPredictor.load("models/ag")
test = pd.read_csv("test.csv")
log_pred = predicator.predict(test)
hours = np.expm1(log_pred).clip(0.1)

out = test[["project_id"]].copy() if "project_id" in test.columns else pd.DataFrame(index=test.index)
out["hours_spent"] = hours
out.to_csv("predictions.csv", index=False)